# Multiple-comparison correction across all flow-asymmetry × anhedonia correlations

**Purpose.** This notebook is the *single source of truth* for the multiplicity correction. It recomputes
every per-subject directed-flow asymmetry measure from the Ceff matrices, correlates each with every
anhedonia scale, and applies the corrections in one place — so the family is defined by the hypothesis,
not by which notebook a test happens to live in.

It consolidates:

- **2 system-level (aggregate) measures** — from `payman_post_analyses_CLEAN.ipynb`
  - `Cortex→Subcortex` normalized top-down dominance (`td_dominance_norm`)
  - `PFC→Striatum` normalized asymmetry (`fs_asym_norm`)
- **3 edge-level measures** — from `frontostriatal_connectivity_OLS.ipynb` (the 3 full-model FDR survivors)
  - `vmPFC–NAc`, `dlPFC–NAc`, `ACC–aPUT` normalized asymmetry (`_norm`)

against **4 scales**: `SHAPS_total`, `TEPS_total`, `TEPS_anticipatory`, `TEPS_consummatory`.

All brain measures use the same normalized index **(A − B) / (A + B)**, A = top-down flow, B = bottom-up,
so they are on a common [-1, 1] scale (positive = top-down dominant). Aggregates sum over the block;
edges average over the block — for equal-size blocks these give the same ratio, and both are reproduced
exactly as in the source notebooks.

### Correction design (see final section for how to report)

| Family | Tests | Rationale |
|---|---|---|
| **Primary** | 5 brain × **SHAPS_total** (5) | SHAPS = a-priori primary hedonic-capacity scale |
| **Secondary** | 5 brain × **TEPS_total** (5) | convergent validity |
| **Specificity** | 5 brain × TEPS antic/consum (10) | which reward component drives it |
| **Flat-20 (robustness)** | all 5 × 4 (20) | maximally conservative, single BH-FDR family |

Two correlation types are computed; **Pearson is primary**, Spearman is reported as a sensitivity check.
Only the primary type is corrected (correcting across both would inflate the family artificially).

## 1 · Setup and paths

In [ ]:
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

BASE  = '/Users/proghani/Documents/personal/local_thesis_proj/TCP_thesis_project/my_analysis/trophic_coherence_analysis'
TCP   = '/Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data'
PHENO = '/Users/proghani/Documents/personal/local_thesis_proj/TCP_thesis_project/phenotype/imputed'

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

## 2 · Load Ceff matrices, region labels, and subject IDs

In [ ]:
# Effective connectivity: Ceff[i, j] is directed edge j -> i  (row = TARGET, col = SOURCE)
data_low  = sio.loadmat(f'{BASE}/outputs/NEW_4_factor_clustering_ipnybV4/low_anhedonia/results_Ceff_low_anhedonia.mat')
data_high = sio.loadmat(f'{BASE}/outputs/NEW_4_factor_clustering_ipnybV4/high_anhedonia/results_Ceff_high_anhedonia.mat')
Ceff_LOW, Ceff_HIGH = data_low['Ceff_LOW'], data_high['Ceff_HIGH']   # (NSUB, 232, 232)
NSUB_LOW, NSUB_HIGH = Ceff_LOW.shape[0], Ceff_HIGH.shape[0]

# Region labels (Schaefer-200 cortex + Tian-S2 subcortex = 232). Odd lines = names.
lines = open(f'{TCP}/atlases/Schaefer2018_200Parcels_7Networks_order_Tian_Subcortex_S2_label.txt').read().strip().splitlines()
region_labels = [lines[i] for i in range(0, len(lines), 2)]
assert len(region_labels) == 232

# Subject IDs (row order matches Ceff_LOW / Ceff_HIGH)
mat_low  = sio.loadmat(f'{TCP}/anhedonia/NEW_4_factor_clustering_ipnybV4/low_anhedonia.mat')
mat_high = sio.loadmat(f'{TCP}/anhedonia/NEW_4_factor_clustering_ipnybV4/high_anhedonia.mat')
ids_low  = [mat_low['low_anhedonia'][i, 1].flat[0]   for i in range(mat_low['low_anhedonia'].shape[0])]
ids_high = [mat_high['high_anhedonia'][i, 1].flat[0] for i in range(mat_high['high_anhedonia'].shape[0])]
ids_all  = ids_low + ids_high
group_all = ['LOW'] * NSUB_LOW + ['HIGH'] * NSUB_HIGH

print(f'NSUB  LOW={NSUB_LOW}  HIGH={NSUB_HIGH}  pooled={NSUB_LOW + NSUB_HIGH}')

## 3 · ROI index definitions
Identical filters to both source notebooks.

In [ ]:
_idx = lambda pred: [i for i, l in enumerate(region_labels) if pred(l)]

# Striatal
NAc_idx  = [8, 9, 24, 25]            # ventral striatum
aPUT_idx = [12, 28]                  # anterior putamen
Caudate  = [14, 15, 30, 31]
Putamen  = [12, 13, 28, 29]

# Cortical (Schaefer-200 7Networks)
mPFC_idx  = _idx(lambda l: 'Default_PFC' in l and 'pCun' not in l)
OFC_idx   = _idx(lambda l: 'Limbic_OFC' in l or 'Cont_OFC' in l)
ACC_idx   = _idx(lambda l: 'SalVentAttn_Med' in l or 'Cont_Cing' in l)
dlPFC_idx = _idx(lambda l: 'Cont_PFCl' in l)

# Compartments / aggregates
cortical_idx    = np.arange(32, 232)          # 200 cortical
subcortical_idx = np.arange(0, 32)            # 32 subcortical
pfc_idx         = dlPFC_idx + mPFC_idx + OFC_idx + ACC_idx
striatum_idx    = NAc_idx + Caudate + Putamen

print('mPFC', len(mPFC_idx), '| OFC', len(OFC_idx), '| ACC', len(ACC_idx), '| dlPFC', len(dlPFC_idx))
print('PFC aggregate', len(pfc_idx), '| striatum aggregate', len(striatum_idx))

## 4 · Per-subject normalized flow-asymmetry measures

Normalized index **(A − B)/(A + B)**, A = top-down (source → target block), B = bottom-up.
Convention `Ceff[target, source]`, so top-down flow PFC→Str = `Ceff[striatal_rows, pfc_cols]`.
Aggregates **sum** over the block; edges take the **mean** — exactly as in the source notebooks
(equal block sizes ⇒ identical ratio, so the two are directly comparable).

In [ ]:
def norm_edge(Cg, tgt, src):
    # Edge measure: mean over blocks. tgt/src = target/source ROI index lists.
    n = Cg.shape[0]; out = np.empty(n)
    for s in range(n):
        A = Cg[s][np.ix_(tgt, src)].mean()   # top-down  (src -> tgt)
        B = Cg[s][np.ix_(src, tgt)].mean()   # bottom-up (tgt -> src)
        out[s] = (A - B) / (A + B)
    return out

def norm_agg(Cg, tgt, src, eps=0.0):
    # Aggregate measure: sum over blocks (eps guards div-by-zero, as in source).
    n = Cg.shape[0]; out = np.empty(n)
    for s in range(n):
        A = Cg[s][np.ix_(tgt, src)].sum()
        B = Cg[s][np.ix_(src, tgt)].sum()
        out[s] = (A - B) / (A + B + eps)
    return out

_cat = lambda lo, hi: np.concatenate([lo, hi])

brain = {}
# Aggregates: top-down = cortex->subcortex  (A = Ceff[subcortical, cortical])
brain['Cortex->Subcortex'] = _cat(norm_agg(Ceff_LOW,  subcortical_idx, cortical_idx, 1e-12),
                                  norm_agg(Ceff_HIGH, subcortical_idx, cortical_idx, 1e-12))
brain['PFC->Striatum']     = _cat(norm_agg(Ceff_LOW,  striatum_idx, pfc_idx),
                                  norm_agg(Ceff_HIGH, striatum_idx, pfc_idx))
# Edges: top-down = PFC->striatal  (A = Ceff[striatal, pfc])
brain['vmPFC-NAc'] = _cat(norm_edge(Ceff_LOW,  NAc_idx,  mPFC_idx),  norm_edge(Ceff_HIGH, NAc_idx,  mPFC_idx))
brain['dlPFC-NAc'] = _cat(norm_edge(Ceff_LOW,  NAc_idx,  dlPFC_idx), norm_edge(Ceff_HIGH, NAc_idx,  dlPFC_idx))
brain['ACC-aPUT']  = _cat(norm_edge(Ceff_LOW,  aPUT_idx, ACC_idx),   norm_edge(Ceff_HIGH, aPUT_idx, ACC_idx))

BRAIN_COLS = list(brain.keys())
brain_df = pd.DataFrame({'subjectkey': ids_all, 'anhedonia_group': group_all, **brain})

print('Per-subject brain-measure means (pooled):')
print(brain_df[BRAIN_COLS].mean().round(4).to_string())
n_bad = brain_df[BRAIN_COLS].isna().sum()
if n_bad.any():
    print('\nUndefined (A+B==0) subjects per measure:'); print(n_bad[n_bad > 0].to_string())

## 5 · Anhedonia scales (SHAPS + TEPS total/subscales)

In [ ]:
shaps_items = pd.read_csv(f'{PHENO}/imputed_shaps01.csv')
shaps_items['SHAPS_total'] = shaps_items.iloc[:, 1:].sum(axis=1)

teps_items = pd.read_csv(f'{PHENO}/imputed_teps01.csv')
ANTICIPATORY = [1, 4, 6, 8, 10, 11, 13, 15, 16, 18]
CONSUMMATORY = [2, 3, 5, 7, 9, 12, 14, 17]
assert sorted(ANTICIPATORY + CONSUMMATORY) == list(range(1, 19))
teps_items['TEPS_total']  = teps_items[[f'teps{i}' for i in range(1, 19)]].sum(axis=1)
teps_items['TEPS_antic']  = teps_items[[f'teps{i}' for i in ANTICIPATORY]].sum(axis=1)
teps_items['TEPS_consum'] = teps_items[[f'teps{i}' for i in CONSUMMATORY]].sum(axis=1)

sid_sh, sid_te = shaps_items.columns[0], teps_items.columns[0]
scales = (shaps_items[[sid_sh, 'SHAPS_total']].rename(columns={sid_sh: 'subjectkey'})
          .merge(teps_items[[sid_te, 'TEPS_total', 'TEPS_antic', 'TEPS_consum']]
                 .rename(columns={sid_te: 'subjectkey'}), on='subjectkey', how='outer'))

SCALE_COLS = ['SHAPS_total', 'TEPS_total', 'TEPS_antic', 'TEPS_consum']
df = brain_df.merge(scales, on='subjectkey', how='inner')
print(f'Merged subjects: {len(df)}  (brain={len(brain_df)}, scales={len(scales)})')

## 6 · All correlations (long format)
Pearson (primary) and Spearman (sensitivity) for every brain × scale pair.

In [ ]:
FAMILY = {'SHAPS_total': 'primary', 'TEPS_total': 'secondary',
          'TEPS_antic': 'specificity', 'TEPS_consum': 'specificity'}

rows = []
for b in BRAIN_COLS:
    for sc in SCALE_COLS:
        d = df[[b, sc]].dropna()
        r_p, p_p = pearsonr(d[b], d[sc])
        r_s, p_s = spearmanr(d[b], d[sc])
        rows.append(dict(brain=b, scale=sc, family=FAMILY[sc], n=len(d),
                         r=r_p, p=p_p, rho=r_s, p_spearman=p_s))
res = pd.DataFrame(rows)
print(f'{len(res)} correlations computed.')

## 7 · Apply the corrections
`q_flat20` = single BH-FDR over all 20 (Pearson). `q_tiered` = BH-FDR within each family.

In [ ]:
res['q_flat20'] = multipletests(res['p'], method='fdr_bh')[1]

res['q_tiered'] = np.nan
for fam, m in res.groupby('family').groups.items():
    res.loc[m, 'q_tiered'] = multipletests(res.loc[m, 'p'], method='fdr_bh')[1]

def stars(q):
    return '***' if q < .001 else '**' if q < .01 else '*' if q < .05 else 'ns'

res['sig_flat20'] = res['q_flat20'].map(stars)
res['sig_tiered'] = res['q_tiered'].map(stars)
res = res.sort_values(['family', 'brain']).reset_index(drop=True)

### 7a · Full consolidated results table

In [ ]:
show = res[['brain', 'scale', 'family', 'n', 'r', 'p', 'q_tiered', 'sig_tiered',
            'q_flat20', 'sig_flat20', 'rho', 'p_spearman']].copy()
for c in ['r', 'p', 'q_tiered', 'q_flat20', 'rho', 'p_spearman']:
    show[c] = show[c].round(4)
show

### 7b · Primary family only (5 brain × SHAPS) — the headline result

In [ ]:
prim = res[res['family'] == 'primary'][['brain', 'n', 'r', 'p', 'q_tiered', 'sig_tiered', 'q_flat20', 'sig_flat20']]
prim.round(4).reset_index(drop=True)

### 7c · Publication matrix — Pearson r with tiered-FDR stars
Rows = brain measure, columns = scale; each cell `r (stars)`.

In [ ]:
def cell(row):
    return f"{row['r']:+.3f} {row['sig_tiered']}"
res['cell'] = res.apply(cell, axis=1)
pub = res.pivot(index='brain', columns='scale', values='cell').reindex(index=BRAIN_COLS, columns=SCALE_COLS)
pub

### 7d · Save the consolidated table to CSV

In [ ]:
out_csv = f'{BASE}/consolidated_correlations_FDR.csv'
res.drop(columns='cell').to_csv(out_csv, index=False)
print('Saved ->', out_csv)

## 8 · How to read this / how to report it

**What survives.**
- **Primary family (× SHAPS):** all five measures are FDR-significant — the two aggregates strongly,
  the three edges at the *p* < .05 / *q* < .05 level. This is the result to lead with.
- **Secondary (× TEPS_total) and specificity (× subscales):** largely non-significant after FDR.
  The **SHAPS-present / TEPS-absent dissociation** is itself informative (construct specificity), not a
  failure — a generic selection artifact would not distinguish the two scales.

**Why the scheme matters (be transparent about this).** The three edges pass under the **tiered** primary
family (5 tests) but sit just above threshold under the **flat-20** correction. Report both columns and
state which is primary; do not present only the one that is kinder to the edges.

**Suggested manuscript sentence.**
> Correlations between the five directed-flow asymmetry measures and anhedonia severity were FDR-corrected
> (Benjamini–Hochberg). The primary family comprised each measure's correlation with SHAPS total (five tests);
> TEPS total and its anticipatory/consummatory subscales were examined as secondary convergent-validity and
> specificity analyses and corrected within their own families. A conservative single-family correction across
> all 20 tests is reported alongside (`q_flat20`). Spearman correlations were concordant.

**Caveats to state.** The 3 frontostriatal edges are components of the PFC→Striatum aggregate, so those
tests are not independent — this makes the FDR conservative rather than anti-conservative. Scales are also
mutually correlated (TEPS_total = anticipatory + consummatory), which is why Bonferroni is inappropriate here.

## 9 · 15-panel scatter grid (SHAPS / TEPS Consummatory / TEPS Anticipatory)

Rows = SHAPS total, TEPS Consummatory, TEPS Anticipatory; columns = the five flow-asymmetry measures in fixed order. Each panel is annotated with Pearson r, raw p, and the FDR q within the manuscript families (SHAPS = primary 5-test family; the two TEPS subscales = specificity 10-test family, pooled). Saves `scatter_15_asym_vs_scales.png`.

In [ ]:
# %% 15-panel grid: 5 flow-asymmetry measures vs SHAPS / TEPS Consummatory / TEPS Anticipatory
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11})
from matplotlib import font_manager as fm
from pathlib import Path

# register the Roboto files matplotlib can't see yet (one-time per kernel)
for f in Path.home().glob('Library/Fonts/Roboto-*.ttf'):
    fm.fontManager.addfont(str(f))

plt.rcParams['font.family'] = 'Roboto'


GROUP_COLORS = {'LOW': '#3AAFB9', 'HIGH': '#D65A7A'}

COLS = BRAIN_COLS  # Cortex->Subcortex, PFC->Striatum, vmPFC-NAc, dlPFC-NAc, ACC-aPUT
ROWS = [('SHAPS_total', 'SHAPS total'),
        ('TEPS_consum', 'TEPS Consummatory'),
        ('TEPS_antic',  'TEPS Anticipatory')]

NICE = {'Cortex->Subcortex': 'Cortex→Subcortex', 'PFC->Striatum': 'PFC→Striatum',
        'vmPFC-NAc': 'vmPFC–NAc', 'dlPFC-NAc': 'dlPFC–NAc', 'ACC-aPUT': 'ACC–aPUT'}

# --- 1. Pearson r, p for all 15 pairs ---
stat = {}
for scale, _ in ROWS:
    for m in COLS:
        d = df[[m, scale]].dropna()
        r, p = pearsonr(d[m], d[scale])
        stat[(scale, m)] = {'r': r, 'p': p, 'n': len(d)}

# --- 2. FDR within the manuscript families ---
sh_keys = [('SHAPS_total', m) for m in COLS]                       # primary: 5 tests
for k, q in zip(sh_keys, multipletests([stat[k]['p'] for k in sh_keys], method='fdr_bh')[1]):
    stat[k]['q'] = q
sp_keys = [(s, m) for s in ('TEPS_consum', 'TEPS_antic') for m in COLS]  # specificity: 10 tests
for k, q in zip(sp_keys, multipletests([stat[k]['p'] for k in sp_keys], method='fdr_bh')[1]):
    stat[k]['q'] = q

pf = lambda v: '<0.001' if v < 0.001 else f'{v:.3f}'

# --- 3. 3x5 grid ---
fig, axes = plt.subplots(len(ROWS), len(COLS),
                         figsize=(5 * len(COLS), 4.5 * len(ROWS)),
                         squeeze=False)

for i, (scale, scale_lab) in enumerate(ROWS):
    for j, m in enumerate(COLS):
        ax = axes[i, j]
        ax.grid(False)
        sub = df[[m, scale, 'anhedonia_group']].dropna()
        x, y = sub[scale].to_numpy(), sub[m].to_numpy()

        for g, c in GROUP_COLORS.items():
            mask = (sub['anhedonia_group'] == g).to_numpy()
            ax.scatter(x[mask], y[mask], s=28, alpha=0.65, color=c,
                       edgecolor='white', linewidth=0.5, label=g)

        # pooled OLS fit line over the whole sample
        b, a = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 100)
        ax.plot(xs, a + b * xs, color='black', lw=1.8)

        st = stat[(scale, m)]
        ax.set_title(f'{NICE[m]}  ·  {scale_lab}', fontsize=12, fontweight='bold')
        ax.set_xlabel(f'{scale_lab}')
        ax.set_ylabel(f'{NICE[m]} (norm)')

        ax.xaxis.set_major_formatter(FormatStrFormatter('%.0f'))
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
        ax.text(0.96, 0.04,
                rf'r = {st["r"]:+.3f}' '\n'
                rf'$p_\mathrm{{raw}}$ = {pf(st["p"])}' '\n'
                rf'$p_\mathrm{{FDR}}$ = {pf(st["q"])}',
                transform=ax.transAxes, va='bottom', ha='right', fontsize=16,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='0.7', alpha=0.85))


# axes[0, -1].legend(title='Anhedonia', frameon=False, loc='lower right')
fig.suptitle('Flow-asymmetry measures vs SHAPS and TEPS subscales — whole sample (N = 210)',
             y=1.00, fontsize=14)
fig.tight_layout()
plt.savefig('scatter_15_asym_vs_scales.png', dpi=600, bbox_inches='tight')
plt.show()
